In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 120

skus = [f'SKU{str(i).zfill(3)}' for i in range(1, n+1)]
categories = np.random.choice(['Electronics', 'Clothing', 'Food'], n, p=[0.35, 0.35, 0.30])
suppliers = np.random.choice(['Supplier A', 'Supplier B', 'Supplier C', 'Supplier D'], n, p=[0.25, 0.30, 0.25, 0.20])
warehouses = np.random.choice(['North', 'South', 'East'], n, p=[0.40, 0.35, 0.25])

stock_units = np.random.randint(0, 500, n)
reorder_point = np.random.randint(20, 100, n)
lead_time = np.random.randint(2, 30, n)
lead_time = lead_time.astype(float)
lead_time[suppliers == 'Supplier D'] = np.random.randint(20, 45, (suppliers == 'Supplier D').sum())

unit_cost = np.round(np.random.uniform(5, 200, n), 2)
units_sold = np.random.randint(10, 300, n)
units_sold[categories == 'Electronics'] = np.random.randint(100, 300, (categories == 'Electronics').sum())

defect_rate = np.round(np.random.uniform(0.01, 0.05, n), 4)
defect_rate[suppliers == 'Supplier C'] = np.round(np.random.uniform(0.05, 0.12, (suppliers == 'Supplier C').sum()), 4)

on_time = np.random.choice(['Yes', 'No'], n, p=[0.80, 0.20])
on_time[suppliers == 'Supplier D'] = np.random.choice(['Yes', 'No'], (suppliers == 'Supplier D').sum(), p=[0.55, 0.45])

df = pd.DataFrame({
    'SKU': skus,
    'Category': categories,
    'Supplier': suppliers,
    'Warehouse': warehouses,
    'Stock_Units': stock_units,
    'Reorder_Point': reorder_point,
    'Lead_Time_Days': lead_time,
    'Unit_Cost': unit_cost,
    'Units_Sold': units_sold,
    'Defect_Rate': defect_rate,
    'On_Time_Delivery': on_time
})

df.loc[np.random.choice(df.index, 5, replace=False), 'Lead_Time_Days'] = np.nan
df.loc[np.random.choice(df.index, 4, replace=False), 'Defect_Rate'] = np.nan
df['Revenue'] = (df['Units_Sold'] * df['Unit_Cost']).round(2)

print(df.shape)
print(df.head())

(120, 12)
      SKU     Category    Supplier Warehouse  Stock_Units  Reorder_Point  \
0  SKU001     Clothing  Supplier D      East          177             72   
1  SKU002         Food  Supplier D      East          162             61   
2  SKU003         Food  Supplier B      East          379             77   
3  SKU004     Clothing  Supplier A     North           32             58   
4  SKU005  Electronics  Supplier A     North          416             33   

   Lead_Time_Days  Unit_Cost  Units_Sold  Defect_Rate On_Time_Delivery  \
0            24.0     140.82          47          NaN              Yes   
1            21.0      85.27         103       0.0310              Yes   
2            14.0     175.49         104       0.0276              Yes   
3             6.0     105.47          78       0.0260               No   
4            24.0     194.76         278       0.0324              Yes   

    Revenue  
0   6618.54  
1   8782.81  
2  18250.96  
3   8226.66  
4  54143.28  


##Which product category is making us the most money?

In [ ]:
#clean
nulos=df.isnull().sum()
duplicados=df.duplicated().sum()
#group category
group=df.groupby('Category')['Revenue'].sum()
total_revenue=group.sum()
print(total_revenue)
#best category
best_cat=group.idxmax()
high_rev=group.max()
print((high_rev/total_revenue*100).round(1))

2106898.74
52.7


###**Conclusion**: Electronics leads all product categories with $1,110,607 in total revenue, representing 52.7% of the business — making it the core revenue driver of the operation.
###**Recomendation**:  Prioritize investment in Electronics inventory. Negotiate with suppliers for better pricing and ensure stock levels meet demand to protect this revenue stream.

##Which products are at risk of running out of stock?

In [ ]:
filtrar=df[df['Stock_Units']<df['Reorder_Point']].copy()
filtrar['Diff']=filtrar['Stock_Units']-filtrar['Reorder_Point']
filtrar['categorize']= filtrar.apply(lambda x: "Critical" if (x['Reorder_Point']>x['Stock_Units']*2) else 'Attention',axis=1)
total_problems=(len(filtrar))
critical=filtrar[filtrar['categorize']=="Critical"]
print(f'Total: {total_problems}')
print(f'critical: {len(critical)}')
print(f'attention: {total_problems-len(critical)}')
ordenar=(filtrar[['SKU','Stock_Units','Reorder_Point','Supplier','Diff','categorize']]).sort_values(by='Diff',ascending=True)
print(ordenar)

Total: 13
critical: 4
attention: 9
        SKU  Stock_Units  Reorder_Point    Supplier  Diff categorize
88   SKU089           18             93  Supplier A   -75   Critical
69   SKU070           36             99  Supplier A   -63   Critical
119  SKU120           38             75  Supplier A   -37  Attention
102  SKU103           51             83  Supplier A   -32  Attention
11   SKU012           64             93  Supplier A   -29  Attention
73   SKU074           12             39  Supplier C   -27   Critical
3    SKU004           32             58  Supplier A   -26  Attention
26   SKU027           34             57  Supplier C   -23  Attention
15   SKU016           42             65  Supplier B   -23  Attention
19   SKU020           11             28  Supplier D   -17   Critical
115  SKU116           52             66  Supplier C   -14  Attention
43   SKU044           74             86  Supplier B   -12  Attention
110  SKU111           24             32  Supplier C    -8  Attention

###Conclusion: Out of 120 SKUs analyzed, 13 products show inventory risk — 4 classified as Critical and 9 requiring attention.
###SKU089 (Supplier A) is the most urgent case, sitting 75 units below its reorder point — the largest gap in the dataset.
###Recommendation: Immediately escalate replenishment orders for Critical SKUs with their respective suppliers. Prioritize SKU089 and establish a weekly stock review for all flagged products.

###Which supplier is causing us the most quality problems?

In [ ]:
#clean
nuls=df.isnull().sum()
#fill empty places with the average
df['Defect_Rate'] = df['Defect_Rate'].fillna(df['Defect_Rate'].mean())
group=filter.groupby('Supplier')['Defect_Rate'].sum().round(2)
orden=group.sort_values(ascending=False)
print(orden)

Supplier
Supplier C    2.73
Supplier A    1.09
Supplier B    0.80
Supplier D    0.64
Name: Defect_Rate, dtype: float64


Missing Defect_Rate values were replaced with the column average to ensure a realistic comparison across suppliers.
###Conclusion: Supplier C shows the highest defect rate at 2.73 — significantly above the rest. Supplier D, by contrast, records the lowest rate at 0.64, making it a more reliable quality partner.
###Recommendation: Schedule an immediate quality review meeting with Supplier C to identify root causes. Evaluate whether Supplier D can absorb some of Supplier C's product lines as a mitigation strategy.

##Which warehouse is the most efficient?

In [ ]:
group = df.groupby('Warehouse').agg(
    Revenue=('Revenue', 'sum'),
    On_Time=('On_Time_Delivery', lambda x: (x == 'Yes').sum()),
    Total=('On_Time_Delivery', 'count')
)
group['On_Time_%'] = (group['On_Time'] / group['Total'] * 100).round(1)
group['Revenue'] = group['Revenue'].round(2)
group = group.drop(columns=['Total'])
group = group.sort_values(by='Revenue', ascending=False)
print(group)

             Revenue  On_Time  On_Time_%
Warehouse                               
North      800730.56       27       61.4
South      782618.53       32       68.1
East       523549.65       22       75.9


###North Warehouse leads in revenue with 800,730.00 but records the lowest on-time delivery rate at 61.4% - a significant operational risk for the highest-volume location.
###South Warehouse shows a balanced profile with 782,618 in revenue and 68.1% on-time delivery, though neither metric reaches an optimal level.
###East Warehouse generates the lowest revenue at 523,549 but delivers the best service level at 75.9% on-time — suggesting stronger operational discipline.
###Recommendation: Investigate the root causes of delivery delays in North Warehouse. Resolving these issues could make it the top performer in both revenue and service — maximizing its full potential.

##Which supplier takes the longest to deliver?

In [ ]:
df['Lead_Time_Days'] = df['Lead_Time_Days'].fillna(df['Lead_Time_Days'].mean())
group=df.groupby('Supplier')['Lead_Time_Days'].mean()
peor_supp=group.idxmax()
peor_lead=group.max()
print(peor_supp)
print(f'{peor_lead:.2f}')
print(group.sort_values(ascending=False).head())


Supplier D
30.28
Supplier
Supplier D    30.282609
Supplier B    15.606061
Supplier C    15.232819
Supplier A    13.517903
Name: Lead_Time_Days, dtype: float64


###Supplier D records the longest average lead time at 30.28 days — more than double compared to the other suppliers, whose averages range between 13.51 and 15.60 days. This gap represents a significant supply chain risk, particularly for high-demand or critical SKUs.
###Recommendation: Review the contract terms with Supplier D and set a maximum acceptable lead time threshold. If performance does not improve, evaluate alternative suppliers for the product lines currently assigned to Supplier D.